# 🚀 Qwen2.5-Coder-32B Web UI Action Prediction Pipeline (src2)

이 노트북은 `src2/` 파이프라인을 사용하여 웹 UI 액션 예측 모델을 학습하고 추론하기 위해 설계되었습니다.

### 📌 권장 사양
- **런타임:** Google Colab **A100 (80GB VRAM)** 권장
- **데이터:** `data/train.csv`, `data/test.csv` 파일이 필요합니다.

## 1. 환경 설정
Unsloth 및 필수 라이브러리를 설치합니다.

In [ ]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    # A100, H100, L4 등 최신 GPU용
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    # T4, V100 등 이전 GPU용
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pandas tqdm beautifulsoup4 lxml scikit-learn

## 2. 프로젝트 코드 및 데이터 준비
로컬에서 작업한 프로젝트 파일을 업로드하고 압축을 해제하거나 경로를 확인합니다.

In [ ]:
import os
# 파일 패널을 통해 프로젝트 압축파일(예: project.zip)을 업로드한 경우 아래 실행
# !unzip project.zip -d .

print(f"현재 작업 경로: {os.getcwd()}")
print("프로젝트 구조 확인:")
!ls -R src2 | head -n 20
!ls data/

## 3. 모델 학습 (Training)
`src2/train.py`를 실행하여 Qwen2.5-Coder-32B 모델을 학습시킵니다.

### 옵션 선택
- **일반 학습:** 빠른 테스트를 위해 사용합니다.
- **OOF 학습 (`--oof`):** 3-Fold 교차 검증을 통해 더 강력한 성능과 앙상블을 구축할 때 사용합니다.

In [ ]:
# [방법 1] 단일 모델 학습
!python src2/train.py

# [방법 2] 3-Fold OOF 학습 (최종 제출용 권장)
# !python src2/train.py --oof

## 4. 추론 및 제출 파일 생성 (Inference)
학습된 LoRA 가중치를 사용하여 `submission.csv`를 생성합니다.

### 옵션 선택
- **단일 추론:** `src2_lora_model`을 사용하여 추론합니다.
- **앙상블 추론 (`--ensemble`):** OOF 학습 결과물인 3개 폴드 모델을 결합합니다.

In [ ]:
# [방법 1] 단일 모델 추론
!python src2/inference.py

# [방법 2] 앙상블 추론 (OOF 학습을 완료한 경우)
# !python src2/inference.py --ensemble

## 5. 결과 다운로드
생성된 결과 파일들을 로컬로 다운로드합니다.

In [ ]:
from google.colab import files

# 제출 파일 다운로드
if os.path.exists("submission.csv"):
    files.download("submission.csv")

# 학습 지표 리포트 다운로드
if os.path.exists("outputs/eval_metrics.json"):
    files.download("outputs/eval_metrics.json")